# Genixcure · 11 PCR samples — full FASTQ reanalysis and QC v2

**Reason:** Previously the exact unique tables were produced with `--disable_quality_filtering`, no primer trimming, and no index removal. `PCR_6_13` produced 6,335,288 unique sequences from 14,567,586 merged reads, unlike the other ten (83k–219k unique). That is a **QC warning, not proof of contamination or failure**.

**This notebook reads the ORIGINAL R1/R2 FASTQ**, applies paired-read quality filtering then `fastp` merging, screens terminal AAV primers (including orientation), and reports *separately* strict both-primer, one-primer, no-primer, ambiguous and too-short categories. Only confident primer ends are removed. Index `NNNNNN` is **not automatically trimmed**. Outputs are newly written under `PCR_11_Reanalysis_QC_v2`, never overwriting original results.

**Initial controls:** `QUALIFIED_BASE_Q=20`, `MAX_LOW_QUALITY_PERCENT=20`, `MIN_MEAN_Q=25`, `MAX_N=0`; these are pilot thresholds, not prevalidated assay cutoffs. Quality filter changes both which read pairs survive and merged sequence statistics. Primer motifs are exact matches in terminal 45 nt; unrecognized/truncated primers remain unresolved, not silently treated as clean.

**Processing logic:** Drive API authenticated download → MD5 verify → fastp QC and merging → terminal primer/orientation classification of merged FASTQ → external sort, weighted exact sequence counts → checksums/count reconciliation → sample-level comparison to previous CSVs. Keep all categories to avoid selection bias. Run **PCR_6_13 first**, then remaining ten; stop and inspect its primer/read-quality metrics before interpreting biological diversity. If you want to enforce a strict gate before running the other samples, set `RUN_ONLY_PCR_6_13=True`.

PacBio BAM is **not** included in this analysis.

**Old results:** before starting fresh QC, a validated copy of the 11 previous result files is created under `Backup_PreReanalysis_2026-11-03`. No original data deletion.


In [ ]:
!command -v fastp >/dev/null || (apt-get update -qq >/dev/null && apt-get install -y -qq fastp >/dev/null)
!fastp --version

In [ ]:
from google.colab import auth,drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from collections import Counter
import google.auth, gzip, csv, json, re, os, hashlib, subprocess, shutil, time, datetime
import pandas as pd

auth.authenticate_user()
drive.mount('/content/drive')
creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
api=build('drive','v3',credentials=creds,cache_discovery=False)

FOLDERS={
'Brain_1':'1DlWiq9dpcvZZ6Xpw8uovd8EhTxTIKRaZ',
'Brain_2':'1uDkNHE_t8Fs6lXtd-Ftbi1SxaOhvK4mf',
'Brain_3':'1woHxLMEJYnw_nF_sE0lVHqhqvTN-fAVq',
'Motor_1':'1yPdiJB5jnAN_lqdrPGUpALbsXm7Q2Jv9',
'Motor_2':'1brTJvspA9Q-RSjPkuEkfoPqu685_9hmL',
'Motor_3':'1tEJLs5pyPrV8nvwH86XGDSWRcUb6YWdJ',
'SH_SY5Y_1':'1mc0tlqGQaiveM3OBpPof__5MoK0EAZfv',
'SH_SY5Y_2':'1sMzD62LwPBFFuHvFvC2cpihhXpf1QyBt',
'SH_SY5Y_3':'12_6fz_2g0OBROfBxi3IsvBkg9Q8AuKmO',
'Input':'1F58gLBc_48r_R2dkRVc8ILvnNJW8Lt1n',
'PCR_6_13':'12YerJ_CIPzXwpjOf6krNy8QKwH_zg1nt'
}
SAMPLES=['PCR_6_13']+[x for x in FOLDERS if x!='PCR_6_13']
RUN_ONLY_PCR_6_13=False  # True: PCR_6_13 1개만 QC 파일럿 실행
OUT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PCR_11_Reanalysis_QC_v2')
OLD=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/All_10_Samples')
BACKUP=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/Backup_PreReanalysis_2026-11-03')
WORK=Path('/content/genixcure_11_reanalysis_tmp')
OUT.mkdir(parents=True,exist_ok=True); WORK.mkdir(parents=True,exist_ok=True)
QUALIFIED_BASE_Q=20
MAX_LOW_QUALITY_PERCENT=20
MIN_MEAN_Q=25
MAX_N=0
MERGED_MIN_MEAN_Q=25       # merged FASTQ에서 별도 품질 재검사
MERGED_MAX_LOW_Q20_PCT=20  # 병합된 최종 Read의 Q20 미만 염기 비율
MERGED_MIN_LENGTH=40
MIN_RAW_READ_LENGTH=40
FASTP_THREADS=4
SORT_RAM='2G'
MAX_END_OVERHANG=45
MIN_INSERT_LENGTH=20
OVERWRITE=False
CATEGORIES=('both_end','single_end','unmatched','ambiguous','too_short')
print('Output:',OUT)

## 0. 이전 결과 백업 (최초 실행)

기존 Drive 파일은 삭제하거나 덮어쓰지 않습니다. 이전 결과 11개와 관련 요약 CSV를 `Backup_PreReanalysis_2026-11-03/All_10_Samples/`에 **검증된 사본으로 보관**합니다. 이후 신버전은 `PCR_11_Reanalysis_QC_v2`에만 기록합니다.

원본 폴더 이동 권한이 연결 앱에서 거부된 경우에도 Colab에서 소유자 권한으로 백업을 수행할 수 있습니다. 충분한 Drive 저장 공간을 확보하세요. 백업이 확인되기 전 새 분석은 진행하지 않습니다.

In [ ]:
import hashlib,shutil
BACKUP_SOURCE=OLD
BACKUP_DEST=BACKUP/'All_10_Samples'
BACKUP.mkdir(parents=True,exist_ok=True)
BACKUP_DEST.mkdir(parents=True,exist_ok=True)
expected_old=[OLD/(s+'_merged_unique_sequences.csv.gz') for s in FOLDERS]
missing=[p.name for p in expected_old if not p.is_file()]
assert not missing, 'Previous results missing; backup not complete: '+str(missing)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

backup_manifest=[]
for src in sorted(OLD.iterdir()):
    if not src.is_file():continue
    dst=BACKUP_DEST/src.name
    original_size=src.stat().st_size
    original_hash=sha256_file(src)
    if not dst.exists() or dst.stat().st_size!=original_size or sha256_file(dst)!=original_hash:
        part=BACKUP_DEST/(src.name+'.partial')
        shutil.copyfile(src,part)
        assert part.stat().st_size==original_size and sha256_file(part)==original_hash
        part.replace(dst)
    backup_manifest.append({'name':src.name,'bytes':original_size,'sha256':original_hash})
# Verify the complete set of old 11 unique CSV results after copy.
assert all((BACKUP_DEST/p.name).is_file() for p in expected_old)
with open(BACKUP/'backup_manifest.json','w') as f:
    json.dump({'files':backup_manifest,'source':str(OLD),'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()},f,indent=2)
print('BACKUP VERIFIED:',len(backup_manifest),'files in',BACKUP_DEST)


## 1. Primer classification self-check

The user-provided primers: `AAV_F=GAAAGGACAGTGGGAGTGGC`, `AAV_R=GCATGGACGAGCTGTACAAG`. The observed reverse-complement of AAV_F is also considered. P5/P7 adapters are reported as possible artifacts; no speculative 6N removal. A 5′ primer can be F or R, a 3′ primer must be reverse-complement of F or R. For **both end** pairs, F...RC(R) or R...RC(F) are considered valid; mismatched pairs are ambiguous. Every confident trimmed sequence is normalized to F→R insert orientation before aggregation. Single-end sequences remain in original orientation and **are not merged with both_end**.

In [ ]:
PRIMERS={'F':'GAAAGGACAGTGGGAGTGGC','R':'GCATGGACGAGCTGTACAAG'}
P5='CACGACGCTCTTCCGATCTAA'
P7='GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
TRANS=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(TRANS)[::-1]

def terminal_hits(s, side):
    hits=[]
    for name,motif in PRIMERS.items():
        p=motif if side=='left' else rc(motif)
        at=s.find(p)
        while at>=0:
            overhang=at if side=='left' else len(s)-(at+len(p))
            if overhang<=MAX_END_OVERHANG:
                hits.append((overhang,name,at,at+len(p)))
            at=s.find(p,at+1)
    hits.sort()
    if not hits:return []
    return [h for h in hits if h[0]==hits[0][0]]

def classify(seq):
    s=seq.upper()
    left=terminal_hits(s,'left');right=terminal_hits(s,'right')
    if len(left)>1 or len(right)>1:return 'ambiguous',s,'multiple_terminal_matches'
    a=left[0] if left else None;b=right[0] if right else None
    if a and b:
        if a[3]>b[2] or a[1]==b[1]:return 'ambiguous',s,'overlap_or_incompatible_ends'
        seq_out=s[a[3]:b[2]]
        if len(seq_out)<MIN_INSERT_LENGTH:return 'too_short',s,'after_both_primer_trim'
        if a[1]=='R':seq_out=rc(seq_out)
        return 'both_end',seq_out,a[1]+'__RC_'+b[1]
    if a or b:
        seq_out=s[a[3]:] if a else s[:b[2]]
        if len(seq_out)<MIN_INSERT_LENGTH:return 'too_short',s,'after_single_primer_trim'
        return 'single_end',seq_out,('left_'+a[1]) if a else ('right_RC_'+b[1])
    return 'unmatched',s,'none'

insert='ACGTACGTACGTACGTACGTACGT'
assert classify(PRIMERS['F']+insert+rc(PRIMERS['R']))[:2]==('both_end',insert)
assert classify(PRIMERS['R']+rc(insert)+rc(PRIMERS['F']))[:2]==('both_end',insert)
assert classify(insert+rc(PRIMERS['F']))[0]=='single_end'
assert classify(insert)[0]=='unmatched'
assert classify(PRIMERS['F']+'AC'+rc(PRIMERS['R']))[0]=='too_short'
print('Primer/orientation unit tests PASS')

## 2. Discover FASTQ, download & integrity check

Only the original raw FASTQ is analyzed. Google Drive files must have matching names ending `_1.fq.gz` and `_2.fq.gz` (or `.fastq.gz`), and their MD5 and byte lengths are checked when available. Local temporary FASTQ are deleted **after validated outputs**; failures retain local files for retry.

In [ ]:
def discover(sample):
    fid=FOLDERS[sample]; items=[];token=None
    while True:
        x=api.files().list(q=f"'{fid}' in parents and trashed=false",
            fields='nextPageToken,files(id,name,size,md5Checksum)',
            pageSize=1000,pageToken=token,supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        items+=x.get('files',[]);token=x.get('nextPageToken')
        if not token:break
    r1=[f for f in items if re.search(r'_1\.(?:fq|fastq)\.gz$',f['name'],re.I)]
    r2=[f for f in items if re.search(r'_2\.(?:fq|fastq)\.gz$',f['name'],re.I)]
    if len(r1)!=1 or len(r2)!=1:raise RuntimeError(f'R1/R2 not unique: {sample} {len(r1)} {len(r2)}')
    prefix=lambda f:re.sub(r'_[12]\.(?:fq|fastq)\.gz$','',f['name'],flags=re.I)
    if prefix(r1[0])!=prefix(r2[0]):raise ValueError('Paired filenames differ')
    return [r1[0],r2[0]]

def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

def download(item,work):
    dest=work/item['name']; expected=int(item['size'])
    if dest.exists() and dest.stat().st_size!=expected:dest.unlink()
    if not dest.exists():
        creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        svc=build('drive','v3',credentials=creds,cache_discovery=False)
        req=svc.files().get_media(fileId=item['id'],supportsAllDrives=True)
        with dest.open('wb') as f:
            dl=MediaIoBaseDownload(f,req,chunksize=16*1024*1024)
            done=False
            while not done:_,done=dl.next_chunk(num_retries=5)
    assert dest.stat().st_size==expected,'Invalid download byte count'
    if item.get('md5Checksum'):assert md5(dest)==item['md5Checksum'],'MD5 mismatch'
    return dest

## 3. Quality filter + overlap merge + primer-aware count

`fastp`: QC **ON**, no automatic adapter trim / no poly-G trim (avoids changing insert ends without checking); paired-read merged output only. Overlap thresholds match earlier analysis (20 nt / at most 5 mismatches / 20%). Separate fastp JSON/HTML report is retained per sample. `--average_qual` is average per read filter, not Q30-everywhere. New filtered results are **not directly comparable** to old unfiltered counts without accounting for filtering losses. The merged read retains its quality string, although the final exact-count files do not.

In [ ]:
def analyze_fastq(merged_path,work):
    temp=work/'tagged.tsv'
    counters=Counter();motifs=Counter()
    n=n_pass=n_lowqual=0
    with gzip.open(merged_path,'rt',newline='') as r,temp.open('w',buffering=1024*1024) as w:
        while True:
            name=r.readline()
            if not name:break
            seq=r.readline().strip();plus=r.readline();qual=r.readline().strip()
            if not plus.startswith('+') or len(seq)!=len(qual):
                raise RuntimeError('Invalid merged FASTQ record')
            if not seq or any(b not in 'ACGTNacgtn' for b in seq):
                raise RuntimeError('Unexpected nucleotide')
            n+=1
            q=[ord(ch)-33 for ch in qual]
            if (len(seq)<MERGED_MIN_LENGTH or sum(q)/len(q)<MERGED_MIN_MEAN_Q
                or 100*sum(x<20 for x in q)>MERGED_MAX_LOW_Q20_PCT*len(q)
                or 'N' in seq.upper()):
                n_lowqual+=1
                continue
            category,trimmed,motif=classify(seq)
            w.write(f'{category}\t{trimmed}\t1\n')
            counters[category]+=1;motifs[(category,motif)]+=1
            n_pass+=1
            if n%1000000==0:
                print(f'    merged scanned={n:,}, QC pass={n_pass:,}, QC fail={n_lowqual:,}',flush=True)
    assert n==n_pass+n_lowqual
    return temp,n,n_pass,n_lowqual,counters,motifs

def write_count_tables(tagged,work,output_dir,sample):
    sorted_path=work/'tagged.sorted.tsv'
    env=dict(os.environ,LC_ALL='C')
    subprocess.run(['sort','--parallel=4','-S',SORT_RAM,'-T',str(work),
                    '-t','\t','-k1,1','-k2,2',str(tagged),'-o',str(sorted_path)],
                   check=True,env=env)
    handles={};counts={x:{'unique':0,'reads':0,'singleton':0} for x in CATEGORIES}
    temp_outputs={cat:work/(sample+'_'+cat+'_unique.csv.gz') for cat in CATEGORIES}
    try:
        for cat,p in temp_outputs.items():
            handles[cat]=gzip.open(p,'wt',compresslevel=1)
            handles[cat].write('sequence,read_count,length_bp\n')
        def emit(cat,seq,ct):
            if not seq:return
            handles[cat].write(f'{seq},{ct},{len(seq)}\n')
            counts[cat]['unique']+=1;counts[cat]['reads']+=ct
            if ct==1:counts[cat]['singleton']+=1
        prev=None;n=0
        with sorted_path.open('r') as fh:
            for line in fh:
                cat,seq,ct=line.rstrip('\n').split('\t');ct=int(ct)
                key=(cat,seq)
                if prev is not None and key!=prev:
                    emit(prev[0],prev[1],n);n=0
                prev=key;n+=ct
            if prev:emit(prev[0],prev[1],n)
    finally:
        for f in handles.values():f.close()
    # Complete integrity scan, including gzip footer; then stage on Drive
    for cat,p in temp_outputs.items():
        nr=total=0;last=None
        with gzip.open(p,'rt') as f:
            reader=csv.DictReader(f)
            assert reader.fieldnames==['sequence','read_count','length_bp']
            for row in reader:
                seq=row['sequence'];ct=int(row['read_count'])
                assert seq and ct>0 and len(seq)==int(row['length_bp'])
                assert last is None or last<seq,'Duplicate sequence within category'
                last=seq;nr+=1;total+=ct
        assert (nr,total)==(counts[cat]['unique'],counts[cat]['reads'])
        final=output_dir/p.name
        staged=output_dir/(p.name+'.partial')
        shutil.copyfile(p,staged)
        assert staged.stat().st_size==p.stat().st_size
        staged.replace(final)
    return counts

def old_reference(sample):
    p=OLD/(sample+'_merged_unique_sequences.csv.gz')
    if not p.exists():
        p=BACKUP_DEST/(sample+'_merged_unique_sequences.csv.gz')
    if not p.exists():return {'old_merged_reads':None,'old_unique_sequences':None}
    unique=total=0
    with gzip.open(p,'rt') as f:
        rd=csv.DictReader(f)
        for x in rd:
            unique+=1;total+=int(x['read_count'])
    return {'old_merged_reads':total,'old_unique_sequences':unique}

def run_sample(sample):
    dest=OUT/sample
    dest.mkdir(parents=True,exist_ok=True)
    report_path=dest/'reanalysis_summary.json'
    if not OVERWRITE and report_path.exists():
        summary=json.loads(report_path.read_text())
        expected=[dest/(sample+'_'+cat+'_unique.csv.gz') for cat in CATEGORIES]
        if all(p.exists() and p.stat().st_size>0 for p in expected):
            print(f'[{sample}] existing report + category tables present; skip (use OVERWRITE=True to rerun)')
            return summary
    w=WORK/sample;w.mkdir(parents=True,exist_ok=True)
    t0=time.monotonic()
    pair=discover(sample)
    print(f'[{sample}] download R1/R2',flush=True)
    with ThreadPoolExecutor(max_workers=2) as pool:
        fq1,fq2=list(pool.map(lambda x:download(x,w),pair))
    print(f'[{sample}] quality filtering and merging',flush=True)
    merged=w/'merged_filtered.fq.gz'
    cmd=['fastp','-i',str(fq1),'-I',str(fq2),'--merge',
        '--merged_out',str(merged),'-o','/dev/null','-O','/dev/stdout',
        '--disable_adapter_trimming','--disable_trim_poly_g',
        '--qualified_quality_phred',str(QUALIFIED_BASE_Q),
        '--unqualified_percent_limit',str(MAX_LOW_QUALITY_PERCENT),
        '--average_qual',str(MIN_MEAN_Q),'--n_base_limit',str(MAX_N),
        '--length_required',str(MIN_RAW_READ_LENGTH),
        '--overlap_len_require','20','--overlap_diff_limit','5',
        '--overlap_diff_percent_limit','20',
        '--thread',str(FASTP_THREADS),'--json',str(w/'fastp.json'),
        '--html',str(w/'fastp.html')]
    with (w/'fastp.log').open('w') as log:
        subprocess.run(cmd,check=True,stdout=subprocess.DEVNULL,stderr=log)
    t_merge=time.monotonic()
    print(f'[{sample}] primer screening, sorting, recounting',flush=True)
    tagged,nmerged,n_qpass,n_qfail,per_category,motifs=analyze_fastq(merged,w)
    stats=write_count_tables(tagged,w,dest,sample)
    assert n_qpass+n_qfail==nmerged
    assert sum(v['reads'] for v in stats.values())==n_qpass,'Read count lost during grouping'
    fp=json.loads((w/'fastp.json').read_text())
    fastp_merged=fp.get('merged_and_filtered',{}).get('total_reads')
    if fastp_merged is not None:assert int(fastp_merged)==nmerged,'Fastp merge count mismatch'
    raw_reads=int(fp['summary']['before_filtering']['total_reads'])
    assert raw_reads%2==0
    summary={'sample':sample,'raw_pairs':raw_reads//2,'fastp_merged_reads':nmerged,'merged_qc_pass':n_qpass,'merged_qc_failed':n_qfail,
             'merged_rate_raw_pct':round(n_qpass/(raw_reads//2)*100,3),
             'fastp_report':str(dest/'fastp.json'),
             'quality':{'qualified_base_q':QUALIFIED_BASE_Q,
                        'max_low_q_percent':MAX_LOW_QUALITY_PERCENT,
                        'min_mean_q':MIN_MEAN_Q,'max_N':MAX_N,
                        'merged_min_mean_q':MERGED_MIN_MEAN_Q,
                        'merged_max_low_q20_pct':MERGED_MAX_LOW_Q20_PCT,
                        'min_read_length':MIN_RAW_READ_LENGTH},
             'category_counts':stats,
             'motif_count_top':[(k[0],k[1],v) for k,v in motifs.most_common(20)],
             'elapsed_seconds':round(time.monotonic()-t0,1),
             **old_reference(sample)}
    shutil.copyfile(w/'fastp.json',dest/'fastp.json')
    shutil.copyfile(w/'fastp.html',dest/'fastp.html')
    (dest/'reanalysis_summary.json.partial').write_text(json.dumps(summary,indent=2))
    (dest/'reanalysis_summary.json.partial').replace(report_path)
    print(f'[{sample}] merged_QC_PASS={n_qpass:,} merged_QC_FAIL={n_qfail:,} both={stats["both_end"]["reads"]:,} '
          f'single={stats["single_end"]["reads"]:,} unmatched={stats["unmatched"]["reads"]:,}',flush=True)
    shutil.rmtree(w,ignore_errors=True)
    return summary

## 4. Execute 11 samples with a progress summary

`PCR_6_13` is processed first. The notebook writes category CSV files and separate fastp reports per sample. A failure does not overwrite old data; the temporary downloaded reads remain for debugging. Review especially how many reads are classified as `both_end` and how many unique/singletons remain after trimming.

In [ ]:
results=[]
for sample in SAMPLES[:1] if RUN_ONLY_PCR_6_13 else SAMPLES:
    try:
        x=run_sample(sample);results.append(x)
    except Exception as exc:
        print(f'[{sample}] ERROR {type(exc).__name__}: {exc}',flush=True)
        results.append({'sample':sample,'error':str(exc)})
    summary_rows=[]
    for x in results:
        row={k:x.get(k) for k in ('sample','raw_pairs','old_merged_reads','old_unique_sequences','fastp_merged_reads','merged_qc_pass','merged_qc_failed','merged_rate_raw_pct','elapsed_seconds','error')}
        for cat in CATEGORIES:
            stat=x.get('category_counts',{}).get(cat,{})
            row.update({f'{cat}_{k}':stat.get(k) for k in ('reads','unique','singleton')})
        summary_rows.append(row)
    pd.DataFrame(summary_rows).to_csv(OUT/'PCR_11_reanalysis_summary.csv',index=False)
display(pd.DataFrame(summary_rows))
print('Saved:',OUT)

## How to interpret results

- **Raw pairs vs QC merged reads:** QC can remove low-quality reads before pairing/merge; differences from original baseline reflect multiple causes (not only primer). `fastp.json` records filtering statistics.
- **`both_end`**: high-confidence two-primer motif reads, insert orientation normalized. Compare **these** unique/singletons across samples while reporting percentage of retained reads.
- **`single_end`**: one verified primer only; trimmed on detected side but not comparable directly to complete inserts; do not merge with both_end.
- **`unmatched` / `ambiguous`**: not assumed clean or defective. They are **not automatically discarded**; retain for root-cause review.
- Exact primers with substitutions or truncation can escape detection; index/barcode positions and amplicon architecture have not been independently validated. A single substitution from sequencing errors can still inflate Unique. For variant inference, consider a subsequent error-aware consensus/UMI analysis once design and primer positions are confirmed.
- Each category preserves the full **read count** from its input, and all categories reconcile to the merged FASTQ count.
- **Do not infer plasmid/cell/molecule count** from this exact-sequence count; PCR duplicates and lack of UMI prevent that inference.